In [ ]:
from modules import GameEncoder
from replay import Replay, iter_replays
from utils import summarise, launched

game = Replay("replay_memory/games/000/g0000042.npz")

model = GameEncoder(32)


In [8]:
t=77
player=1
game.meta["agents"], len(game)
# game.planets(t)             # (n, 7) [id, owner, x, y, radius, ships, production]
game.fleets(t).shape             # (m, 7) [id, owner, x, y, angle, from_planet_id, ships]
# game.moves(t, player)       # [[planet_id, angle, ships], ...] as the agent sent them
# game.observation(t, player) # Kaggle-style observation dict (works with features.encode)
# game.arrays                 # the raw arrays above, for vectorised use

(38, 7)

In [13]:
# for t in range(len(game)):
#     print(game.planets(t).shape)

In [11]:
import torch  
import torch.nn as nn  
import torch.optim as optim  
import math  

In [14]:
import json, os, numpy as np

ROOT = "replay_memory"
games = sorted((json.loads(line) for line in open(f"{ROOT}/index.jsonl")), key=lambda g: g["index"])

def game_path(g):           # adjust if `file` is stored differently (e.g. already a full path)
    return os.path.join(ROOT, g["file"])

counts, n_players = [], []
for g in games:
    with np.load(game_path(g)) as z:
        counts.append(z["fleet_n"])
    n_players.append(np.full(len(counts[-1]), len(g["agents"])))   # one agent per seat
counts, n_players = np.concatenate(counts), np.concatenate(n_players)

for n in (2, 4):
    c = counts[n_players == n]
    print(f"{n}p: {len(c)} states, mean {c.mean():.1f}, max {c.max()}, empty {(c == 0).mean():.1%}, "
          f"p50/p90/p99/p99.9 {np.percentile(c, [50, 90, 99, 99.9])}")

2p: 1312256 states, mean 41.0, max 730, empty 4.5%, p50/p90/p99/p99.9 [ 21.  97. 307. 458.]
4p: 2181093 states, mean 33.5, max 735, empty 3.1%, p50/p90/p99/p99.9 [ 24.  70. 192. 346.]
